In [ ]:
import os

# Either correct the variable …
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5 pyspark-shell"
)

# … or simply remove it if you’re already passing the package via builder.config:
# os.environ.pop("PYSPARK_SUBMIT_ARGS", None)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("VehicleLocationDataValidation")
    .master("local[*]")                         # optional but explicit
    .config(
        "spark.jars",
        "/Users/gyauk/codetools/javajdbc/postgresql-42.7.5.jar",
    )
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5",
    )
    .getOrCreate()
)

print("Spark version:", spark.version)


25/06/22 12:29:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.5.5


25/06/22 15:04:14 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 451237 ms exceeds timeout 120000 ms
25/06/22 15:04:15 WARN SparkContext: Killing executors is not supported by current scheduler.
25/06/22 15:04:15 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, to_timestamp, to_date, year, current_date, when, regexp_extract, upper, lower, ltrim, rtrim, length, isnull, isnan, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, DoubleType, BooleanType

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("UserRentalDataValidation") \
    .getOrCreate()
      # .config("spark.driver.extraClassPath", "/path/to/hadoop-aws.jar:/path/to/aws-java-sdk-bundle.jar") 

# --- Define Schemas ---

# User Data Schema
user_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("driver_license_number", StringType(), True),
    StructField("driver_license_expiry", StringType(), True), # Read as String for initial validation
    StructField("creation_date", StringType(), True), # Read as String for initial validation
    StructField("is_active", IntegerType(), True) # Assuming 0 or 1
])

# Rental Transactions Data Schema
rental_schema = StructType([
    StructField("rental_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("rental_start_time", StringType(), True), # Read as String for initial validation
    StructField("rental_end_time", StringType(), True),   # Read as String for initial validation
    StructField("pickup_location", IntegerType(), True),
    StructField("dropoff_location", IntegerType(), True),
    StructField("total_amount", DoubleType(), True)
])


25/06/22 10:47:28 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [18]:

# --- Read Data ---

# Local file paths (ensure these CSV files are in your project directory or provide full paths)
local_user_data_path = "/Users/gyauk/Desktop/DataEngineering/phase_2/Lab4/users.csv"
local_rental_data_path = "/Users/gyauk/Desktop/DataEngineering/phase_2/Lab4/rental_transactions.csv"

# S3 paths (commented out as requested for local testing)
# s3_user_data_path = "s3a://your-s3-bucket-name/users.csv"
# s3_rental_data_path = "s3a://your-s3-bucket-name/rental_transactions.csv"

print("Reading user data from local file...")
try:
    df_users = spark.read \
        .option("header", "true") \
        .schema(user_schema) \
        .csv(local_user_data_path)
    print("User data schema after initial read:")
    df_users.printSchema()
    df_users.show(5, truncate=False)
except Exception as e:
    print(f"Error reading local user data: {e}")
    df_users = None

print("\nReading rental transaction data from local file...")
try:
    df_rentals = spark.read \
        .option("header", "true") \
        .schema(rental_schema) \
        .csv(local_rental_data_path)
    print("Rental data schema after initial read:")
    df_rentals.printSchema()
    df_rentals.show(5, truncate=False)
except Exception as e:
    print(f"Error reading local rental data: {e}")
    df_rentals = None


Reading user data from local file...
User data schema after initial read:
root
 |-- user_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- driver_license_number: string (nullable = true)
 |-- driver_license_expiry: string (nullable = true)
 |-- creation_date: string (nullable = true)
 |-- is_active: integer (nullable = true)

+----------+----------+---------+--------------------------+------------------+---------------------+---------------------+-------------+---------+
|user_id   |first_name|last_name|email                     |phone_number      |driver_license_number|driver_license_expiry|creation_date|is_active|
+----------+----------+---------+--------------------------+------------------+---------------------+---------------------+-------------+---------+
|26d08ab733|Lisa      |Parker   |lisa.parker@gmail.com     |334.271.2972x60554|M

In [19]:

# ---Basic Validations and Transformations (User Data) ---

if df_users:
    print("\n--- Validating and Transforming User Data ---")

    # Null Checks and Handling
    print("\nChecking for nulls in critical user columns...")
    critical_user_cols = ["user_id", "email", "phone_number", "creation_date", "is_active"]
    for col_name in critical_user_cols:
        null_count = df_users.filter(col(col_name).isNull()).count()
        if null_count > 0:
            print(f"WARNING: Column '{col_name}' has {null_count} null values.")
            df_users = df_users.na.drop(subset=[col_name])
            print(f"Dropped rows with nulls in '{col_name}'. Remaining rows: {df_users.count()}")

    # Data Type Conversions and Formatting
    print("\nPerforming data type conversions and formatting for user data...")

    # Convert date columns
    df_users_transformed = df_users.withColumn("driver_license_expiry", to_date(col("driver_license_expiry"), "yyyy-MM-dd")) \
                                   .withColumn("creation_date", to_date(col("creation_date"), "yyyy-MM-dd"))

    # Validate date conversions
    df_users_transformed = df_users_transformed.withColumn("driver_license_expiry_valid",
        when(col("driver_license_expiry").isNull(), lit(False)).otherwise(lit(True)))
    df_users_transformed = df_users_transformed.withColumn("creation_date_valid",
        when(col("creation_date").isNull(), lit(False)).otherwise(lit(True)))

    # invalid_dl_expiry = df_users_transformed.filter(col("driver_license_expiry_valid") == False).count()
    # if invalid_dl_expiry > 0:
    #     print(f"WARNING: {invalid_dl_expiry} invalid 'driver_license_expiry' values detected after conversion.")
    # invalid_creation_date = df_users_transformed.filter(col("creation_date_valid") == False).count()
    # if invalid_creation_date > 0:
    #     print(f"WARNING: {invalid_creation_date} invalid 'creation_date' values detected after conversion.")

    # # Validate 'is_active' column: should be 0 or 1
    # df_users_transformed = df_users_transformed.withColumn("is_active_valid",
    #     when((col("is_active") == 0) | (col("is_active") == 1), lit(True)).otherwise(lit(False)))
    # invalid_active_users = df_users_transformed.filter(col("is_active_valid") == False).count()
    # if invalid_active_users > 0:
    #     print(f"WARNING: {invalid_active_users} invalid 'is_active' values (not 0 or 1) in users data.")
    #     df_users_transformed = df_users_transformed.withColumn("is_active",
    #         when(col("is_active_valid"), col("is_active")).otherwise(lit(None).cast(IntegerType())))

    # # Email format validation (simple regex, can be more complex if needed)
    # # A common regex for email: `^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$`
    # email_regex = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"
    # df_users_transformed = df_users_transformed.withColumn("email_valid",
    #     when(col("email").rlike(email_regex), lit(True)).otherwise(lit(False)))
    # invalid_emails = df_users_transformed.filter(col("email_valid") == False).count()
    # if invalid_emails > 0:
    #     print(f"WARNING: {invalid_emails} invalid 'email' format detected.")

    # Phone number validation (simple regex for common formats including extensions)
    # This regex attempts to cover formats like ###.###.####x####, (###)###-####, etc.
    # It's a pragmatic regex, not exhaustive for all global formats.
    phone_regex = r"^(\+?\d{1,3}[-. ]?)?\(?\d{3}\)?[-. ]?\d{3}[-. ]?\d{4}(x\d+)?$"
    df_users_transformed = df_users_transformed.withColumn("phone_number_valid",
        when(col("phone_number").rlike(phone_regex), lit(True)).otherwise(lit(False)))
    invalid_phone_numbers = df_users_transformed.filter(col("phone_number_valid") == False).count()
    if invalid_phone_numbers > 0:
        print(f"WARNING: {invalid_phone_numbers} invalid 'phone_number' format detected.")

    # Trim whitespace from string columns
    trim_cols = ["first_name", "last_name", "email", "phone_number", "driver_license_number"]
    for col_name in trim_cols:
        df_users_transformed = df_users_transformed.withColumn(col_name, ltrim(rtrim(col(col_name))))

    # Example: Standardize first_name and last_name to all lower
    df_users_transformed = df_users_transformed.withColumn("first_name", lower(col("first_name"))) \
                                                .withColumn("last_name", lower(col("last_name")))

    print("\nSchema after user transformations:")
    df_users_transformed.printSchema()
    print("\nSample User Data after transformations:")
    df_users_transformed.select("user_id", "first_name", "last_name", "email", "phone_number","phone_number_valid", "driver_license_expiry", "creation_date", "is_active").show(5, truncate=False)




--- Validating and Transforming User Data ---

Checking for nulls in critical user columns...

Performing data type conversions and formatting for user data...

Schema after user transformations:
root
 |-- user_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- driver_license_number: string (nullable = true)
 |-- driver_license_expiry: date (nullable = true)
 |-- creation_date: date (nullable = true)
 |-- is_active: integer (nullable = true)
 |-- driver_license_expiry_valid: boolean (nullable = false)
 |-- creation_date_valid: boolean (nullable = false)
 |-- phone_number_valid: boolean (nullable = false)


Sample User Data after transformations:
+----------+----------+---------+--------------------------+------------------+------------------+---------------------+-------------+---------+
|user_id   |first_name|last_name|email               

In [20]:

# --- 4. Basic Validations and Transformations (Rental Transaction Data) ---

if df_rentals:
    print("\n--- Validating and Transforming Rental Transaction Data ---")

    # 4.1. Null Checks and Handling
    print("\nChecking for nulls in critical rental columns...")
    critical_rental_cols = ["rental_id", "user_id", "vehicle_id", "rental_start_time", "rental_end_time", "total_amount"]
    for col_name in critical_rental_cols:
        null_count = df_rentals.filter(col(col_name).isNull()).count()
        if null_count > 0:
            print(f"WARNING: Column '{col_name}' has {null_count} null values.")
            df_rentals = df_rentals.na.drop(subset=[col_name])
            print(f"Dropped rows with nulls in '{col_name}'. Remaining rows: {df_rentals.count()}")

    # 4.2. Data Type Conversions and Formatting
    print("\nPerforming data type conversions and formatting for rental data...")

    # Convert timestamp columns
    df_rentals_transformed = df_rentals.withColumn("rental_start_time", to_timestamp(col("rental_start_time"), "yyyy-MM-dd HH:mm:ss")) \
                                       .withColumn("rental_end_time", to_timestamp(col("rental_end_time"), "yyyy-MM-dd HH:mm:ss"))

    # Validate timestamp conversions
    # df_rentals_transformed = df_rentals_transformed.withColumn("rental_start_time_valid",
    #     when(col("rental_start_time").isNull(), lit(False)).otherwise(lit(True)))
    # df_rentals_transformed = df_rentals_transformed.withColumn("rental_end_time_valid",
    #     when(col("rental_end_time").isNull(), lit(False)).otherwise(lit(True)))

    # invalid_start_time = df_rentals_transformed.filter(col("rental_start_time_valid") == False).count()
    # if invalid_start_time > 0:
    #     print(f"WARNING: {invalid_start_time} invalid 'rental_start_time' values detected after conversion.")
    # invalid_end_time = df_rentals_transformed.filter(col("rental_end_time_valid") == False).count()
    # if invalid_end_time > 0:
    #     print(f"WARNING: {invalid_end_time} invalid 'rental_end_time' values detected after conversion.")

    # Logical validation: rental_start_time < rental_end_time
    # df_rentals_transformed = df_rentals_transformed.withColumn("rental_duration_valid",
    #     when((col("rental_start_time").isNotNull()) & (col("rental_end_time").isNotNull()) & \
    #          (col("rental_start_time") < col("rental_end_time")), lit(True)).otherwise(lit(False)))

    # invalid_duration = df_rentals_transformed.filter(col("rental_duration_valid") == False).count()
    # if invalid_duration > 0:
    #     print(f"WARNING: {invalid_duration} rentals have invalid durations (start >= end) or null timestamps.")
        # Option: set invalid durations to null or flag them for further investigation
        # df_rentals_transformed = df_rentals_transformed.withColumn("rental_end_time",
        #    when(col("rental_duration_valid"), col("rental_end_time")).otherwise(lit(None).cast(TimestampType())))

    # Validate total_amount: should be non-negative
    # df_rentals_transformed = df_rentals_transformed.withColumn("total_amount_valid",
    #     when(col("total_amount") >= 0, lit(True)).otherwise(lit(False)))
    # invalid_total_amount = df_rentals_transformed.filter(col("total_amount_valid") == False).count()
    # if invalid_total_amount > 0:
    #     print(f"WARNING: {invalid_total_amount} rentals have invalid 'total_amount' (negative).")
    #     # Option: set invalid amounts to null or 0
    #     df_rentals_transformed = df_rentals_transformed.withColumn("total_amount",
    #         when(col("total_amount_valid"), col("total_amount")).otherwise(lit(0.0).cast(DoubleType())))


    print("\nSchema after rental transformations:")
    df_rentals_transformed.printSchema()
    print("\nSample Rental Transaction Data after transformations:")
    df_rentals_transformed.select("rental_id", "user_id", "vehicle_id", "rental_start_time", "rental_end_time", "total_amount").show(5, truncate=False)


# Stop Spark Session
spark.stop()
print("\nSpark Session stopped.")


--- Validating and Transforming Rental Transaction Data ---

Checking for nulls in critical rental columns...

Performing data type conversions and formatting for rental data...

Schema after rental transformations:
root
 |-- rental_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- rental_start_time: timestamp (nullable = true)
 |-- rental_end_time: timestamp (nullable = true)
 |-- pickup_location: integer (nullable = true)
 |-- dropoff_location: integer (nullable = true)
 |-- total_amount: double (nullable = true)


Sample Rental Transaction Data after transformations:
+----------+----------+----------+-------------------+-------------------+------------+
|rental_id |user_id   |vehicle_id|rental_start_time  |rental_end_time    |total_amount|
+----------+----------+----------+-------------------+-------------------+------------+
|b139d8e1b2|320be8068b|0d52304987|2024-02-28 08:05:00|2024-03-01 05:05:00|450.0       |
|7afd